In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
from collections import Counter
from tqdm import tqdm

import shutil
import yaml
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# ============================================================
# PATHS
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("../")

# ============================================================
# PATHS
# ============================================================

SOURCE_DATASET = (
    PROJECT_ROOT /
    "datasets" /
    "processed" /
    "dataset_final"
)

FINAL_DATASET = (
    PROJECT_ROOT /
    "datasets" /
    "processed" /
    "dataset_final"
)

In [3]:
# ============================================================
# CREATE DIRECTORY STRUCTURE
# ============================================================

splits = ["train", "valid", "test"]

for split in splits:

    (FINAL_DATASET / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (FINAL_DATASET / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )

print("Directory structure created.")

Directory structure created.


In [4]:
# ============================================================
# BOUNDING BOX CLEANING RULES
# ============================================================

# Remove extremely tiny boxes
MIN_WIDTH = 0.01
MIN_HEIGHT = 0.01

# Remove extremely tiny areas
MIN_AREA = 0.0001

# Remove absurd aspect ratios
MAX_ASPECT_RATIO = 15

print("Cleaning rules loaded.")

Cleaning rules loaded.


In [5]:
# ============================================================
# CLEAN LABEL FILES
# ============================================================

removed_boxes = 0
kept_boxes = 0

splits = ["train", "valid", "test"]

for split in splits:

    print(f"\n{'='*60}")
    print(f"CLEANING {split.upper()} SET")
    print(f"{'='*60}")

    image_src = SOURCE_DATASET / split / "images"
    label_src = SOURCE_DATASET / split / "labels"

    image_dst = FINAL_DATASET / split / "images"
    label_dst = FINAL_DATASET / split / "labels"

    label_files = list(label_src.glob("*.txt"))

    for label_file in tqdm(label_files):

        cleaned_lines = []

        with open(label_file, "r") as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            if len(parts) != 5:
                removed_boxes += 1
                continue

            class_id = int(parts[0])

            x_center = float(parts[1])
            y_center = float(parts[2])

            width = float(parts[3])
            height = float(parts[4])

            # ------------------------------------------------
            # CLEANING RULES
            # ------------------------------------------------

            area = width * height

            aspect_ratio = max(
                width / (height + 1e-6),
                height / (width + 1e-6)
            )

            invalid = False

            # Tiny boxes
            if width < MIN_WIDTH:
                invalid = True

            if height < MIN_HEIGHT:
                invalid = True

            # Tiny area
            if area < MIN_AREA:
                invalid = True

            # Extreme aspect ratio
            if aspect_ratio > MAX_ASPECT_RATIO:
                invalid = True

            if invalid:
                removed_boxes += 1
                continue

            cleaned_lines.append(line.strip())
            kept_boxes += 1

        # ----------------------------------------------------
        # SKIP EMPTY LABEL FILES
        # ----------------------------------------------------

        if len(cleaned_lines) == 0:
            continue

        # ----------------------------------------------------
        # SAVE CLEANED LABEL FILE
        # ----------------------------------------------------

        output_label = label_dst / label_file.name

        with open(output_label, "w") as f:
            f.write("\n".join(cleaned_lines))

        # ----------------------------------------------------
        # COPY IMAGE
        # ----------------------------------------------------

        image_extensions = [
            ".jpg",
            ".jpeg",
            ".png"
        ]

        for ext in image_extensions:

            image_path = image_src / f"{label_file.stem}{ext}"

            if image_path.exists():

                shutil.copy(
                    image_path,
                    image_dst / image_path.name
                )

                break

print("\nCleaning completed.")


CLEANING TRAIN SET


100%|██████████████████████████████████████████████████████████████████████████████| 2599/2599 [02:01<00:00, 21.41it/s]



CLEANING VALID SET


100%|████████████████████████████████████████████████████████████████████████████████| 104/104 [00:04<00:00, 23.19it/s]



CLEANING TEST SET


100%|██████████████████████████████████████████████████████████████████████████████████| 74/74 [00:03<00:00, 20.67it/s]


Cleaning completed.


In [7]:
# ============================================================
# CLEANING SUMMARY
# ============================================================

print(f"\nRemoved Boxes : {removed_boxes}")
print(f"Kept Boxes    : {kept_boxes}")

removal_ratio = removed_boxes / (removed_boxes + kept_boxes)

print(f"\nRemoval Ratio : {removal_ratio:.4f}")


Removed Boxes : 2019
Kept Boxes    : 26498

Removal Ratio : 0.0708


In [8]:
# ============================================================
# VERIFY FINAL DATASET
# ============================================================

for split in ["train", "valid", "test"]:

    image_count = len(
        list((FINAL_DATASET / split / "images").glob("*"))
    )

    label_count = len(
        list((FINAL_DATASET / split / "labels").glob("*"))
    )

    print(f"\n{split.upper()}")
    print(f"Images : {image_count}")
    print(f"Labels : {label_count}")


TRAIN
Images : 2599
Labels : 2599

VALID
Images : 104
Labels : 104

TEST
Images : 74
Labels : 74
